In [25]:
# Imports -------------------------------------------------------------------------------------
from collections import defaultdict
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
Image.MAX_IMAGE_PIXELS = None

from copy import deepcopy
from io import BytesIO

In [2]:
# OMERO Connection ----------------------------------------------------------------------------
from omero.gateway import BlitzGateway
from omero.model import PolygonI, RoiI
from omero.rtypes import rstring

conn = BlitzGateway("sduenweg", "Jasper100308#", host="wss://wsi.lavlab.mcw.edu/omero-wss", port=443, secure=True)
conn.connect()
conn.c.enableKeepAlive(60000000)

In [3]:
# Configuration -------------------------------------------------------------------------------
# Step 1 Config ====================================================
CSV_PATH = "/Volumes/Siren/Prostate_Analyses/SD_Assorted/2025/Prostate/AnnotationDifference/reannot_key-2.csv"
ANNOTATION_PATH = Path("/Volumes/Siren/Prostate_data/1106/Hist/6/Huron/N106_S06_HE_annot2.jpeg")

# Define Omero Color Key
COLOR_PALETTE = {
    (25,20,255): "Seminal Vesicles",
    (0,0,0): "Atrophy",
    (255,122,0): "HGPIN",
    (48,255,50): "G3",
    (255,250,20): "G4NC",
    (254,22,255): "G4CG",
    (33,255,255): "G5",
}

COLOR_TOLERANCE = 10
MIN_AREA = 25 # adjust as needed

# Polygon simplification
SIMPLIFY_MODE = "relative"
SIMPLIFY_EPSILON = 0.001

# Debugging parameters
SAVE_PREVIEW = False
SHOW_PREVIEW = False

# Visulization parameters
# Maximum width or height of each preview panel.
PREVIEW_MAX_DIMENSION = 1800
PREVIEW_DPI = 120
DRAW_ROI_LABELS = True

# Define configuration dictionaries
EXTRACTION_CONFIG = {
    "color_palette": COLOR_PALETTE,
    "color_tolerance": COLOR_TOLERANCE,
    "min_area": MIN_AREA,
    "simplify_mode": SIMPLIFY_MODE,
    "simplify_epsilon": SIMPLIFY_EPSILON,
}

PREVIEW_CONFIG = {
    "save_preview": SAVE_PREVIEW,
    "show_preview": SHOW_PREVIEW,
    "max_dimension": PREVIEW_MAX_DIMENSION,
    "dpi": PREVIEW_DPI,
    "draw_roi_labels": DRAW_ROI_LABELS,
}

# Step 2 Config ====================================================
TARGET_IMAGE_ID = 9667
ALLOW_COORDINATE_SCALING = True


In [4]:
# Image Utilities -----------------------------------------------------------------------------
def read_annotation(path):
    """
    Load an annotation image as an RGB uint8 NumPy array.

    Parameters
    ----------
    path : str or pathlib.Path
        Path to the annotation JPEG.

    Returns
    -------
    numpy.ndarray
        RGB image with shape (height, width, 3).
    """
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Annotation image not found: {path}")

    try:
        annotation = np.asarray(Image.open(path).convert("RGB"))
    except Exception as exc:
        raise RuntimeError(
            f"Could not read annotation image: {path}"
        ) from exc

    if annotation.ndim != 3 or annotation.shape[2] != 3:
        raise ValueError(
            "Expected an RGB image with shape (height, width, 3), "
            f"but received {annotation.shape}."
        )

    if annotation.dtype != np.uint8:
        annotation = annotation.astype(np.uint8)

    height, width, channels = annotation.shape

    print("=" * 100)
    print("Loaded annotation")
    print(f"Path:     {path}")
    # print(f"Width:    {width:,} pixels")
    # print(f"Height:   {height:,} pixels")
    # print(f"Channels: {channels}")
    # print(f"Data type: {annotation.dtype}")
    print("-" * 70)

    return annotation
    
def color_mask(image, target_color, tolerance):
    """
    Create a binary mask for pixels near a target RGB color.

    A pixel is accepted when every RGB channel differs from the target
    by no more than the specified tolerance.

    Parameters
    ----------
    image : numpy.ndarray
        RGB uint8 image.
    target_color : tuple[int, int, int]
        Target RGB color.
    tolerance : int
        Maximum difference allowed in each RGB channel.

    Returns
    -------
    numpy.ndarray
        Boolean mask with shape (height, width).
    """
    if tolerance < 0:
        raise ValueError("Color tolerance cannot be negative.")

    target = np.asarray(target_color, dtype=np.int16)

    if target.shape != (3,):
        raise ValueError(
            f"Expected an RGB tuple with three values, got {target_color}."
        )

    difference = np.abs(image.astype(np.int16) - target)

    return np.all(difference <= tolerance, axis=2)

def color_mask(image, target_color, tolerance):
    """
    Create a binary mask for pixels near a target RGB color.

    A pixel is accepted when every RGB channel differs from the target
    by no more than the specified tolerance.

    Parameters
    ----------
    image : numpy.ndarray
        RGB uint8 image.
    target_color : tuple[int, int, int]
        Target RGB color.
    tolerance : int
        Maximum difference allowed in each RGB channel.

    Returns
    -------
    numpy.ndarray
        Boolean mask with shape (height, width).
    """
    if tolerance < 0:
        raise ValueError("Color tolerance cannot be negative.")

    target = np.asarray(target_color, dtype=np.int16)

    if target.shape != (3,):
        raise ValueError(
            f"Expected an RGB tuple with three values, got {target_color}."
        )

    difference = np.abs(image.astype(np.int16) - target)

    return np.all(difference <= tolerance, axis=2)

def calculate_centroid(contour):
    """
    Calculate a contour centroid.

    Returns
    -------
    tuple[float | None, float | None]
        Centroid coordinates. Returns (None, None) for a degenerate contour.
    """
    moments = cv2.moments(contour)

    if moments["m00"] == 0:
        return None, None

    cx = moments["m10"] / moments["m00"]
    cy = moments["m01"] / moments["m00"]

    return float(cx), float(cy)

def simplify_contour(contour):
    """
    Simplify a contour using cv2.approxPolyDP.

    Returns
    -------
    numpy.ndarray
        Simplified OpenCV contour.
    """
    if SIMPLIFY_MODE == "fixed":
        epsilon = float(SIMPLIFY_EPSILON)

    elif SIMPLIFY_MODE == "relative":
        perimeter = cv2.arcLength(contour, closed=True)
        epsilon = float(SIMPLIFY_EPSILON) * perimeter

    else:
        raise ValueError(
            "SIMPLIFY_MODE must be either 'fixed' or 'relative'."
        )

    return cv2.approxPolyDP(
        contour,
        epsilon=epsilon,
        closed=True,
    )

In [5]:
# ROI Extraction ------------------------------------------------------------------------------

def extract_rois(annotation, extraction_config):
    """
    Extract polygon contours for each annotation color.

    Parameters
    ----------
    annotation : numpy.ndarray
        RGB annotation image.
    extraction_config : dict
        Configuration dictionary containing:
            - color_palette: dict[tuple[int, int, int], str]
                Mapping of RGB colors to class labels.
            - color_tolerance: int
                Maximum difference allowed in each RGB channel.
            - min_area: float
                Minimum area for a contour to be accepted.
            - simplify_mode: str
                Either "fixed" or "relative" for contour simplification.
            - simplify_epsilon: float
                Epsilon value for contour simplification.

    Returns
    -------
    tuple[list[dict], list[dict]]
        rois:
            One dictionary per accepted ROI.

        class_diagnostics:
            Per-class information including pixel counts and contour
            filtering statistics.
    """
    rois = []
    class_diagnostics = []

    print("\nExtracting ROIs")
    print("-" * 70)

    for color, label in extraction_config["color_palette"].items():
        mask = color_mask(
            image=annotation,
            target_color=color,
            tolerance=extraction_config["color_tolerance"],
        )

        matching_pixels = int(np.count_nonzero(mask))

        contours, _ = cv2.findContours(
            mask.astype(np.uint8),
            mode=cv2.RETR_EXTERNAL,
            method=cv2.CHAIN_APPROX_SIMPLE,
        )

        raw_contour_count = len(contours)
        area_rejected_count = 0
        invalid_polygon_count = 0
        accepted_count = 0

        print(f"{label}")

        # print("-" * 40)
        # print(f"RGB color:             {color}")
        # print(f"Matching pixels:       {matching_pixels:,}")
        # print(f"Raw contours:          {raw_contour_count:,}")

        for raw_contour in contours:
            raw_area = float(cv2.contourArea(raw_contour))

            if raw_area < MIN_AREA:
                area_rejected_count += 1
                continue

            simplified_contour = simplify_contour(raw_contour)

            # A valid polygon needs at least three vertices.
            if len(simplified_contour) < 3:
                invalid_polygon_count += 1
                continue

            simplified_area = float(
                cv2.contourArea(simplified_contour)
            )

            cx, cy = calculate_centroid(simplified_contour)

            rois.append(
                {
                    "id": None,
                    "label": label,
                    "color": tuple(int(value) for value in color),
                    "contour": simplified_contour,
                    "area": simplified_area,
                    "raw_area": raw_area,
                    "centroid": (cx, cy),
                    "raw_vertices": int(len(raw_contour)),
                    "vertices": int(len(simplified_contour)),
                }
            )

            accepted_count += 1

        class_diagnostics.append(
            {
                "label": label,
                "color": color,
                "matching_pixels": matching_pixels,
                "raw_contours": raw_contour_count,
                "area_rejected": area_rejected_count,
                "invalid_polygons": invalid_polygon_count,
                "accepted": accepted_count,
            }
        )

        # print(f"Rejected by area:      {area_rejected_count:,}")
        # print(f"Invalid after simplify:{invalid_polygon_count:>7,}")
        # print(f"Accepted ROIs:         {accepted_count:,}")

    rois = sort_and_assign_ids(rois)

    print(f"Total accepted ROIs: {len(rois):,}")
    return rois, class_diagnostics


def sort_and_assign_ids(rois):
    """
    Sort ROIs consistently and assign class-specific IDs.

    Example IDs:
        G3-01
        G3-02
        G4NC-01
    """
    def sort_key(roi):
        cx, cy = roi["centroid"]

        # Place invalid centroids at the end.
        sort_x = float("inf") if cx is None else cx
        sort_y = float("inf") if cy is None else cy

        return (
            roi["label"],
            sort_y,
            sort_x,
        )

    sorted_rois = sorted(rois, key=sort_key)

    label_counts = defaultdict(int)

    for roi in sorted_rois:
        label = roi["label"]
        label_counts[label] += 1

        roi["id"] = f"{label}-{label_counts[label]:02d}"

    return sorted_rois

In [6]:
# Reporting -----------------------------------------------------------------------------------

def summarize_rois(rois):
    """
    Print ROI count, area, and vertex statistics by annotation class.
    """
    # print("\nROI Summary")
    # print("-" * 70)

    if not rois:
        print("No ROIs were detected.")
        return

    grouped = defaultdict(list)

    for roi in rois:
        grouped[roi["label"]].append(roi)

    header = (
        f"{'Class':<18}"
        f"{'Count':>8}"
        f"{'Mean area':>14}"
        f"{'Min area':>14}"
        f"{'Max area':>14}"
        f"{'Mean vertices':>17}"
        f"{'Max vertices':>15}"
    )

    # print(header)
    # print("-" * len(header))

    for label in sorted(grouped):
        class_rois = grouped[label]

        areas = np.asarray(
            [roi["area"] for roi in class_rois],
            dtype=float,
        )

        vertices = np.asarray(
            [roi["vertices"] for roi in class_rois],
            dtype=float,
        )

        # print(
        #     f"{label:<18}"
        #     f"{len(class_rois):>8,}"
        #     f"{np.mean(areas):>14,.1f}"
        #     f"{np.min(areas):>14,.1f}"
        #     f"{np.max(areas):>14,.1f}"
        #     f"{np.mean(vertices):>17,.1f}"
        #     f"{int(np.max(vertices)):>15,}"
        # )

    # print("-" * len(header))
    # print(f"{'TOTAL':<18}{len(rois):>8,}")


def print_roi_details(rois, maximum=None):
    """
    Print details for individual ROIs.

    Parameters
    ----------
    rois : list[dict]
        Extracted ROI records.
    maximum : int or None
        Maximum number of ROIs to print. None prints every ROI.
    """
    # print("\nIndividual ROI Details")
    # print("=" * 100)

    selected_rois = rois if maximum is None else rois[:maximum]

    for roi in selected_rois:
        cx, cy = roi["centroid"]

        if cx is None:
            centroid_text = "undefined"
        else:
            centroid_text = f"({cx:.1f}, {cy:.1f})"

        # print(
        #     f"{roi['id']:<14}"
        #     f"area={roi['area']:>12,.1f}   "
        #     f"centroid={centroid_text:<24}"
        #     f"vertices={roi['vertices']:>6,}   "
        #     f"raw vertices={roi['raw_vertices']:>6,}"
        # )

    if maximum is not None and len(rois) > maximum:
        print(
            f"Displayed {maximum} of {len(rois)} total ROIs."
        )
    # print("-" * 70)

In [7]:
# Visualization -------------------------------------------------------------------------------

def preview_rois(
    annotation,
    rois,
    source_path=None,
    save_preview=True,
    show_preview=True,
    max_dimension=PREVIEW_MAX_DIMENSION,
    draw_labels=DRAW_ROI_LABELS,
):
    """
    Create a memory-efficient, downsampled visualization.

    Important:
        Downsampling affects only the preview. The original annotation and
        original-resolution ROI contours remain unchanged for later upload.
    """
    height, width = annotation.shape[:2]

    scale = min(
        1.0,
        max_dimension / max(height, width),
    )

    preview_width = max(1, int(round(width * scale)))
    preview_height = max(1, int(round(height * scale)))

    # print("\nCreating preview")
    # print("-" * 70)
    # print(f"Original size: {width:,} × {height:,}")
    # print(
    #     f"Preview size:  {preview_width:,} × "
    #     f"{preview_height:,}"
    # )
    # print(f"Preview scale: {scale:.4f}")

    # INTER_AREA is well suited to shrinking images.
    annotation_preview = cv2.resize(
        annotation,
        dsize=(preview_width, preview_height),
        interpolation=cv2.INTER_AREA,
    )

    figure, axes = plt.subplots(
        nrows=1,
        ncols=2,
        figsize=(12, 6),
        dpi=PREVIEW_DPI,
    )

    axes[0].imshow(annotation_preview)
    axes[0].set_title("Original Annotation — Downsampled")
    axes[0].axis("off")

    # Avoid allocating another full-size RGB array.
    axes[1].set_facecolor("white")
    axes[1].set_xlim(0, preview_width)
    axes[1].set_ylim(preview_height, 0)
    axes[1].set_aspect("equal")

    for roi in rois:
        contour = roi["contour"]

        # Scale only the coordinates used for visualization.
        x = contour[:, 0, 0].astype(np.float32) * scale
        y = contour[:, 0, 1].astype(np.float32) * scale

        x_closed = np.append(x, x[0])
        y_closed = np.append(y, y[0])

        normalized_color = (
            np.asarray(roi["color"], dtype=np.float32) / 255.0
        )

        axes[1].plot(
            x_closed,
            y_closed,
            color=normalized_color,
            linewidth=0.8,
        )

        if draw_labels:
            cx, cy = roi["centroid"]

            if cx is not None and cy is not None:
                axes[1].text(
                    cx * scale,
                    cy * scale,
                    roi["id"],
                    fontsize=4,
                    color="black",
                    ha="center",
                    va="center",
                )

    settings_text = (
        f"ROIs: {len(rois):,} | "
        f"Preview scale: {scale:.3f} | "
        f"Tolerance: ±{COLOR_TOLERANCE} | "
        f"Minimum area: {MIN_AREA:g} px²"
    )

    axes[1].set_title(
        "Extracted Polygon Outlines — Downsampled\n"
        + settings_text
    )
    axes[1].axis("off")

    figure.tight_layout()

    preview_path = None

    if save_preview:
        if source_path is None:
            preview_path = Path("roi_preview.jpg")
        else:
            source_path = Path(source_path)
            preview_path = source_path.with_name(
                f"{source_path.stem}_roi_preview.jpg"
            )

        figure.savefig(
            preview_path,
            dpi=PREVIEW_DPI,
            bbox_inches="tight",
            facecolor="white",
            pil_kwargs={
                "quality": 85,
                "optimize": True,
            },
        )

        print(f"Saved preview: {preview_path}")

    if show_preview:
        plt.show()

    # Explicit cleanup is important in notebooks.
    plt.close(figure)
    del annotation_preview

    return preview_path

In [ ]:
# Coordinate Transform & Scaling --------------------------------------------------------------
def build_coordinate_transform(
    annotation_shape,
    target_width,
    target_height,
):
    """
    Build the coordinate transform from annotation-image space
    to target-image space.
    """

    if len(annotation_shape) < 2:
        raise ValueError(
            "annotation_shape must contain at least height and width."
        )

    annotation_height = int(annotation_shape[0])
    annotation_width = int(annotation_shape[1])

    if annotation_width <= 0 or annotation_height <= 0:
        raise ValueError(
            "Annotation dimensions must be positive."
        )

    if target_width <= 0 or target_height <= 0:
        raise ValueError(
            "Target dimensions must be positive."
        )

    scale_x = target_width / annotation_width
    scale_y = target_height / annotation_height

    dimensions_match = (
        annotation_width == target_width
        and annotation_height == target_height
    )

    return {
        "annotation_width": annotation_width,
        "annotation_height": annotation_height,
        "target_width": int(target_width),
        "target_height": int(target_height),
        "scale_x": scale_x,
        "scale_y": scale_y,
        "dimensions_match": dimensions_match,
        "requires_scaling": not dimensions_match,
    }

def scale_contour(contour, scale_x, scale_y):
    """
    Scale an OpenCV-style contour.

    Supports contours shaped (N, 1, 2) or (N, 2). The original contour
    array is not modified.
    """
    contour = np.asarray(contour)

    if contour.ndim < 2 or contour.shape[-1] != 2:
        raise ValueError(
            "contour must end with an XY coordinate dimension of size 2. "
            f"Received shape {contour.shape}."
        )

    scaled_contour = contour.astype(np.float64, copy=True)

    scaled_contour[..., 0] *= float(scale_x)
    scaled_contour[..., 1] *= float(scale_y)

    return scaled_contour

def scale_roi(roi, scale_x, scale_y):
    """
    Return a copy of one extracted ROI with its geometry scaled.

    Parameters
    ----------
    roi : dict
        ROI dictionary returned by extract_rois().

    scale_x : float
        Horizontal coordinate scaling factor.

    scale_y : float
        Vertical coordinate scaling factor.

    Returns
    -------
    dict
        A copied ROI dictionary containing scaled geometry.
    """
    required_keys = {
        "contour",
        "centroid",
        "area",
        "raw_area",
    }

    missing_keys = required_keys.difference(roi)

    if missing_keys:
        raise KeyError(
            f"ROI is missing required keys: {sorted(missing_keys)}"
        )

    scaled_roi = deepcopy(roi)

    scaled_roi["contour"] = scale_contour(
        contour=roi["contour"],
        scale_x=scale_x,
        scale_y=scale_y,
    )

    cx, cy = roi["centroid"]

    scaled_roi["centroid"] = (
        float(cx) * float(scale_x),
        float(cy) * float(scale_y),
    )

    area_scale = float(scale_x) * float(scale_y)

    scaled_roi["area"] = float(roi["area"]) * area_scale
    scaled_roi["raw_area"] = float(roi["raw_area"]) * area_scale

    return scaled_roi

In [ ]:
def step1_extract_rois(annotation_path, extraction_config, preview_config):
    """
    Extract ROIs from the annotation image.

    Parameters
    ----------
    annotation_path : str or pathlib.Path
        Path to the annotation JPEG.
    extraction_config : dict
        Configuration parameters for ROI extraction.
    preview_config : dict
        Configuration parameters for preview generation.

    Returns
    -------
    tuple[list[dict], list[dict]]
        rois:
            One dictionary per accepted ROI.

        class_diagnostics:
            Per-class information including pixel counts and contour filtering statistics.
    """

    annotation_path = Path(annotation_path)
    if not annotation_path.exists():
        raise FileNotFoundError(f"Annotation image not found: {annotation_path}")

    annotation_img = read_annotation(annotation_path)
    if annotation_img is None:
        raise RuntimeError(f"Failed to read annotation image: {annotation_path}")

    rois, class_diagnostics = extract_rois(
        annotation_img, extraction_config)

    summarize_rois(rois)

    # Change maximum=None to print all ROI details.
    print_roi_details(
        rois,
        maximum=25,
    )

    preview_rois(
        annotation=annotation_img,
        rois=rois,
        source_path=annotation_path,
        save_preview=preview_config["save_preview"],
        show_preview=preview_config["show_preview"],
    )

    if not rois:
        print(
            "\nWARNING: No ROIs were extracted. Check the color palette, "
            "color tolerance, annotation path, and minimum area."
        )

    summary = {
        "annotation_shape": annotation_img.shape,
        "annotation_path": str(annotation_path),
        "rois": rois,
        "class_diagnostics": class_diagnostics,
    }
    
    return summary

In [10]:
def step2_prepare_target_image(
    conn,
    target_image_id,
    summary,
    *,
    allow_coordinate_scaling=False,
):
    """
    Retrieve and validate the target OMERO image.

    Parameters
    ----------
    conn
        Active OMERO BlitzGateway connection.

    target_image_id : int
        OMERO image ID for the target image.

    summary : dict
        Output returned by step1_extract_rois().

    allow_coordinate_scaling : bool, optional
        Whether ROI coordinates may be scaled when annotation and target
        dimensions differ.

    Returns
    -------
    target_context : dict
        Target-image information and coordinate-transform details.
    """

    # Validate the Step 1 summary
    required_keys = {
        "annotation_shape",
        "annotation_path",
        "rois",
        "class_diagnostics",
    }

    missing_keys = required_keys.difference(summary)

    if missing_keys:
        raise KeyError(
            "Step 1 summary is missing required key(s): "
            f"{sorted(missing_keys)}"
        )

    # Validate the OMERO connection
    if conn is None:
        raise ValueError(
            "No OMERO connection was provided."
        )

    if not conn.isConnected():
        raise ConnectionError(
            "The OMERO connection is not active."
        )

    # Retrieve the target image
    target_image = conn.getObject(
        "Image",
        int(target_image_id),
    )

    if target_image is None:
        raise ValueError(
            f"Could not retrieve OMERO image {target_image_id}. "
            "Check that the image exists and that you have permission "
            "to access it."
        )

    target_width = int(target_image.getSizeX())
    target_height = int(target_image.getSizeY())

    # Build the coordinate transform
    transform = build_coordinate_transform(
        annotation_shape=summary["annotation_shape"],
        target_width=target_width,
        target_height=target_height,
    )

    if (
        transform["requires_scaling"]
        and not allow_coordinate_scaling
    ):
        raise ValueError(
            "Annotation and target image dimensions do not match.\n"
            f"Annotation image: "
            f"{transform['annotation_width']} × "
            f"{transform['annotation_height']}\n"
            f"Target OMERO image: "
            f"{transform['target_width']} × "
            f"{transform['target_height']}\n"
            "Coordinate scaling is currently disabled."
        )

    # --------------------------------------------------------
    # Build the Step 2 result
    target_context = {
        "image": target_image,
        "image_id": int(target_image.getId()),
        "image_name": target_image.getName(),
        "transform": transform,
    }

    # Report the result
    print("\n" + "=" * 100)
    print("Step 2 complete")
    print("-" * 70)
    print(
        f"Target image: "
        f"{target_context['image_name']} "
        f"(ID {target_context['image_id']})"
    )
    print(
        f"Annotation dimensions: "
        f"{transform['annotation_width']} × "
        f"{transform['annotation_height']}"
    )
    print(
        f"Target dimensions:     "
        f"{transform['target_width']} × "
        f"{transform['target_height']}"
    )

    if transform["dimensions_match"]:
        print("Coordinate spaces match.")
    else:
        print(
            "Coordinate scaling enabled: "
            f"x={transform['scale_x']:.6f}, "
            f"y={transform['scale_y']:.6f}"
        )

    return target_context

In [ ]:
def step3_transform_rois(summary, target_context):
    """
    Scale extracted ROI geometry into the target image coordinate system.

    Parameters
    ----------
    summary : dict
        Output from step1_extract_rois().

    target_context : dict
        Output from step2_prepare_target_image(). Its "transform"
        dictionary must contain "scale_x" and "scale_y".

    Returns
    -------
    dict
        Context containing the transformed ROIs, target information,
        and transformation metadata.
    """
    if not isinstance(summary, dict):
        raise TypeError("summary must be a dictionary.")

    if not isinstance(target_context, dict):
        raise TypeError("target_context must be a dictionary.")

    required_summary_keys = {
        "annotation_shape",
        "annotation_path",
        "rois",
        "class_diagnostics",
    }

    missing_summary_keys = required_summary_keys.difference(summary)

    if missing_summary_keys:
        raise KeyError(
            "summary is missing required keys: "
            f"{sorted(missing_summary_keys)}"
        )

    required_target_keys = {
        "image",
        "image_id",
        "image_name",
        "transform",
    }

    missing_target_keys = required_target_keys.difference(target_context)

    if missing_target_keys:
        raise KeyError(
            "target_context is missing required keys: "
            f"{sorted(missing_target_keys)}"
        )

    transform = target_context["transform"]

    if not isinstance(transform, dict):
        raise TypeError(
            "target_context['transform'] must be a dictionary."
        )

    required_transform_keys = {"scale_x", "scale_y"}
    missing_transform_keys = required_transform_keys.difference(transform)

    if missing_transform_keys:
        raise KeyError(
            "target_context['transform'] is missing required keys: "
            f"{sorted(missing_transform_keys)}"
        )

    scale_x = float(transform["scale_x"])
    scale_y = float(transform["scale_y"])

    if scale_x <= 0 or scale_y <= 0:
        raise ValueError(
            "scale_x and scale_y must both be greater than zero."
        )

    transformed_rois = [
        scale_roi(
            roi=roi,
            scale_x=scale_x,
            scale_y=scale_y,
        )
        for roi in summary["rois"]
    ]

    step3_context = {
        "image": target_context["image"],
        "image_id": target_context["image_id"],
        "image_name": target_context["image_name"],
        "transform": transform,
        "annotation_shape": summary["annotation_shape"],
        "annotation_path": summary["annotation_path"],
        "rois": transformed_rois,
        "roi_count": len(transformed_rois),
        "class_diagnostics": summary["class_diagnostics"],
    }

    print("\n" + "=" * 100)
    print("Step 3 complete")
    print("-" * 70)
    print(
    f"Transformed {step3_context['roi_count']} ROIs "
    f"for target image "
    f"{step3_context['image_name']} "
    f"(ID: {step3_context['image_id']})"
    )
    
    return step3_context

In [ ]:
# Validate Step3 Transformations ----------------------------------------------------------
def validate_step3_transform(
    summary,
    step3_context,
    atol=1e-6,
):
    """
    Numerically validate the ROI scaling performed during Step 3.

    Raises an AssertionError if a transformed ROI does not match the
    expected scaled coordinates.
    """
    original_rois = summary["rois"]
    transformed_rois = step3_context["rois"]

    transform = step3_context["transform"]
    scale_x = float(transform["scale_x"])
    scale_y = float(transform["scale_y"])

    if len(original_rois) != len(transformed_rois):
        raise AssertionError(
            "ROI count changed during Step 3: "
            f"{len(original_rois)} original versus "
            f"{len(transformed_rois)} transformed."
        )

    area_scale = scale_x * scale_y

    for index, (original, transformed) in enumerate(
        zip(original_rois, transformed_rois)
    ):
        original_contour = np.asarray(
            original["contour"],
            dtype=float,
        )

        transformed_contour = np.asarray(
            transformed["contour"],
            dtype=float,
        )

        expected_contour = original_contour.copy()
        expected_contour[..., 0] *= scale_x
        expected_contour[..., 1] *= scale_y

        if original_contour.shape != transformed_contour.shape:
            raise AssertionError(
                f"ROI {index}: contour shape changed from "
                f"{original_contour.shape} to "
                f"{transformed_contour.shape}."
            )

        if not np.allclose(
            transformed_contour,
            expected_contour,
            atol=atol,
        ):
            max_error = np.max(
                np.abs(transformed_contour - expected_contour)
            )

            raise AssertionError(
                f"ROI {index}: contour coordinates are incorrect. "
                f"Maximum coordinate error: {max_error}"
            )

        original_cx, original_cy = original["centroid"]
        transformed_cx, transformed_cy = transformed["centroid"]

        expected_centroid = (
            float(original_cx) * scale_x,
            float(original_cy) * scale_y,
        )

        if not np.allclose(
            transformed["centroid"],
            expected_centroid,
            atol=atol,
        ):
            raise AssertionError(
                f"ROI {index}: centroid is incorrect. "
                f"Expected {expected_centroid}, "
                f"received {transformed['centroid']}."
            )

        expected_area = float(original["area"]) * area_scale
        expected_raw_area = (
            float(original["raw_area"]) * area_scale
        )

        if not np.isclose(
            transformed["area"],
            expected_area,
            atol=atol,
        ):
            raise AssertionError(
                f"ROI {index}: scaled area is incorrect."
            )

        if not np.isclose(
            transformed["raw_area"],
            expected_raw_area,
            atol=atol,
        ):
            raise AssertionError(
                f"ROI {index}: scaled raw area is incorrect."
            )

        for unchanged_key in (
            "id",
            "label",
            "color",
            "raw_vertices",
            "vertices",
        ):
            if transformed[unchanged_key] != original[unchanged_key]:
                raise AssertionError(
                    f"ROI {index}: '{unchanged_key}' changed "
                    "during transformation."
                )

    print(
        f"Step 3 numerical validation passed for "
        f"{len(transformed_rois)} ROIs."
    )

def validate_roi_bounds(step3_context):
    """
    Check whether transformed ROI coordinates lie within the target image.
    """
    target_image = step3_context["image"]

    target_width = int(target_image.getSizeX())
    target_height = int(target_image.getSizeY())

    out_of_bounds = []

    for index, roi in enumerate(step3_context["rois"]):
        contour = np.asarray(
            roi["contour"],
            dtype=float,
        )

        x = contour[..., 0]
        y = contour[..., 1]

        if (
            np.any(x < 0)
            or np.any(x >= target_width)
            or np.any(y < 0)
            or np.any(y >= target_height)
        ):
            out_of_bounds.append(
                {
                    "roi_index": index,
                    "label": roi["label"],
                    "x_range": (
                        float(np.min(x)),
                        float(np.max(x)),
                    ),
                    "y_range": (
                        float(np.min(y)),
                        float(np.max(y)),
                    ),
                }
            )

    if out_of_bounds:
        print(
            f"Warning: {len(out_of_bounds)} ROI(s) contain "
            "coordinates outside the target image."
        )

        for item in out_of_bounds[:10]:
            print(
                f"  ROI {item['roi_index']} "
                f"({item['label']}): "
                f"x={item['x_range']}, "
                f"y={item['y_range']}"
            )
    else:
        print(
            f"All {len(step3_context['rois'])} ROIs are "
            "within the target image bounds."
        )

    return out_of_bounds

def preview_transformed_rois_thumbnail(
    step3_context,
    max_rois=None,
    linewidth=0.8,
    show_centroids=False,
):
    """
    Preview transformed ROIs over an OMERO-rendered thumbnail.

    The ROIs remain stored in full-resolution target coordinates.
    Coordinates are scaled to thumbnail dimensions only for plotting.

    Parameters
    ----------
    step3_context : dict
        Output from step3_transform_rois().

    max_rois : int or None
        Optional maximum number of ROIs to draw.

    linewidth : float
        Width of plotted contour lines.

    show_centroids : bool
        Whether to display centroid markers.
    """
    target_image = step3_context["image"]

    target_width = int(target_image.getSizeX())
    target_height = int(target_image.getSizeY())

    # Retrieve a rendered thumbnail instead of the full pixel plane.
    thumbnail_bytes = target_image.getThumbnail()
    thumbnail = Image.open(BytesIO(thumbnail_bytes)).convert("RGB")
    thumbnail_array = np.asarray(thumbnail)

    thumbnail_width, thumbnail_height = thumbnail.size

    preview_scale_x = thumbnail_width / target_width
    preview_scale_y = thumbnail_height / target_height

    rois = step3_context["rois"]

    if max_rois is not None:
        rois = rois[:max_rois]

    fig, ax = plt.subplots(figsize=(12, 12))

    ax.imshow(thumbnail_array)

    plotted_count = 0

    for roi in rois:
        contour = np.asarray(
            roi["contour"],
            dtype=float,
        ).reshape(-1, 2)

        if len(contour) < 2:
            continue

        preview_contour = contour.copy()
        preview_contour[:, 0] *= preview_scale_x
        preview_contour[:, 1] *= preview_scale_y

        preview_contour = np.vstack(
            [preview_contour, preview_contour[0]]
        )

        color = np.asarray(
            roi["color"],
            dtype=float,
        ) / 255.0

        ax.plot(
            preview_contour[:, 0],
            preview_contour[:, 1],
            color=color,
            linewidth=linewidth,
        )

        if show_centroids:
            cx, cy = roi["centroid"]

            ax.plot(
                cx * preview_scale_x,
                cy * preview_scale_y,
                marker="x",
                color=color,
                markersize=4,
            )

        plotted_count += 1

    ax.set_title(
        f"Step 3 Thumbnail Preview\n"
        f"{step3_context['image_name']} "
        f"(ID: {step3_context['image_id']})\n"
        f"{plotted_count} ROI(s) displayed"
    )

    ax.set_xlim(0, thumbnail_width)
    ax.set_ylim(thumbnail_height, 0)
    ax.set_aspect("equal")
    ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# Main Execution -----------------------------------------------------------------------------
summary = step1_extract_rois(ANNOTATION_PATH, EXTRACTION_CONFIG, PREVIEW_CONFIG)

target_context = step2_prepare_target_image(
    conn = conn,
    target_image_id=TARGET_IMAGE_ID,
    summary=summary,
    allow_coordinate_scaling=ALLOW_COORDINATE_SCALING,
)

step3_context = step3_transform_rois(
    summary=summary,
    target_context=target_context,
)

# # Run Step 3 Validation
# validate_step3_transform(
#     summary=summary,
#     step3_context=step3_context,
# )

# out_of_bounds = validate_roi_bounds(step3_context)

# preview_transformed_rois_thumbnail(
#     step3_context,
#     show_centroids=False,
# )